# Final post-cluster profile analysis

This notebook profiles the seven retained dimension-specific cluster models without
rerunning clustering or changing any assignment. It uses readable survey labels for
interpretation and internal values only for documented ordered/count comparisons.

Safeguards:

- joins use `respondent_id` and are asserted one-to-one;
- strict final-path cohorts must equal D=191, E=34, F=81 and G=56;
- every model excludes its own source variables and derived components;
- structural non-applicability is never recoded as No;
- conditional variables use eligible-response denominators;
- effect size, coverage and programme relevance drive ranking;
- statistical tests are supporting evidence only;
- no programme recommendations are created.

In [1]:
from pathlib import Path
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = Path.cwd()
OUT = ROOT / "outputs"
LABELS_PATH = OUT / "nysc_survey_cleaned_labels.csv"
VALUES_PATH = OUT / "nysc_survey_cleaned_values.csv"
COMPARISON_OUT = OUT / "final_cluster_profile_comparison_summary.csv"
PROFILE_OUT = OUT / "final_cluster_programme_relevant_profiles.csv"
REPORT_OUT = ROOT / "final_cluster_profiling_report.md"

EXPECTED_BRANCH_COUNTS = {"D": 191, "E": 34, "F": 81, "G": 56}
STRICT_ROUTING_SOURCE = "translated_final_active_branch"
RANDOM_SEED = 20260825


def spec(
    variable,
    label,
    kind="categorical",
    theme="other",
    relevance="medium",
    conditional=False,
    gateway="",
    circularity="",
):
    return {
        "variable": variable,
        "variable_label": label,
        "kind": kind,
        "theme": theme,
        "programme_relevance": relevance,
        "conditional": bool(conditional),
        "gateway": gateway,
        "circularity": circularity,
    }


# Curated, theory- and programme-relevant profiling variables. A variable may form a
# different retained dimension; such cases are explicitly marked as cross-dimensional.
COMMON_DEMOGRAPHICS = [
    spec("age_group", "Age group", theme="equity", relevance="low"),
    spec("gender", "Gender", theme="equity", relevance="low"),
    spec("area_type", "Area type", theme="access", relevance="medium"),
    spec("district", "District", theme="access", relevance="medium", circularity="Many district categories can create sparse tables."),
    spec("moved_from_origin", "Lives in the same area where they grew up", theme="mobility", relevance="medium"),
    spec("move_reason", "Reason for moving", theme="mobility", relevance="medium", conditional=True, gateway="moved_from_origin"),
    spec("marital_status", "Marital status", theme="feasibility", relevance="low"),
    spec("has_children", "Has children", theme="feasibility", relevance="medium"),
]

D_ASPIRATIONS = [
    spec("most_important_life_goal", "Most important life goal", theme="aspiration", relevance="high"),
    spec("most_useful_quality_for_job", "Quality considered most useful for a good job", theme="aspiration", relevance="medium"),
    spec("edu_min_for_decent_job", "Minimum education believed necessary for a decent job", kind="ordered", theme="education", relevance="medium"),
    spec("study_field_preference", "Current or preferred field of study", theme="aspiration", relevance="high"),
    spec("plans_after_education", "Plans after current education", theme="aspiration", relevance="high"),
    spec("ideal_work_type", "Ideal work type", theme="aspiration", relevance="high"),
    spec("ideal_sector", "Ideal sector", theme="aspiration", relevance="high"),
    spec("already_job_searching", "Already searching for work/business", theme="employment", relevance="high"),
    spec("job_search_method_d", "Main job-search method while studying", theme="service", relevance="high", conditional=True, gateway="already_job_searching"),
]

D_NETWORK = [
    spec("employment_network_quality", "Useful employment/training contacts", kind="ordered", theme="support", relevance="high"),
    spec("social_capital_in_edu_access", "How current programme was accessed", theme="service", relevance="medium"),
]

D_EDUCATION = [
    spec("father_edu", "Father's education", kind="ordered", theme="education", relevance="low", circularity="Forms the separate D educational/cultural model."),
    spec("mother_edu", "Mother's education", kind="ordered", theme="education", relevance="low", circularity="Forms the separate D educational/cultural model."),
    spec("expected_highest_edu_d", "Expected highest education", kind="ordered", theme="education", relevance="high", circularity="Forms the separate D educational/cultural model."),
    spec("institution_type", "Current institution type", theme="education", relevance="high", circularity="Forms the separate D educational/cultural model."),
    spec("worked_while_studying", "Worked while studying", theme="employment", relevance="high", circularity="Forms the separate D educational/cultural model."),
    spec("re_entered_education", "Re-entered education after interruption", theme="education", relevance="high", circularity="Forms the separate D educational/cultural model."),
    spec("reason_re_entered", "Reason for re-entering education", theme="education", relevance="high", conditional=True, gateway="re_entered_education", circularity="Follow-up to a variable forming the D educational/cultural model."),
    spec("work_while_study_type", "Type of work while studying", theme="employment", relevance="high", conditional=True, gateway="worked_while_studying", circularity="Conditional on a variable forming the D educational/cultural model."),
    spec("work_while_study_paid", "Paid status of work while studying", theme="employment", relevance="medium", conditional=True, gateway="worked_while_studying", circularity="Conditional on a variable forming the D educational/cultural model."),
    spec("work_while_study_hours", "Hours band while studying", kind="ordered", theme="feasibility", relevance="medium", conditional=True, gateway="worked_while_studying", circularity="Conditional on a variable forming the D educational/cultural model."),
    spec("work_while_study_motivation", "Reason for working while studying", theme="financial", relevance="high", conditional=True, gateway="worked_while_studying", circularity="Conditional on a variable forming the D educational/cultural model."),
]

D_ECONOMIC = [
    spec("household_income", "Household income band", kind="ordered", theme="financial", relevance="high", circularity="Forms the separate D economic model."),
    spec("household_size", "Household size", kind="numeric", theme="financial", relevance="medium", circularity="Forms the separate D economic model."),
    spec("household_workers", "Household workers", kind="numeric", theme="financial", relevance="medium", circularity="Forms the separate D economic model."),
    spec("household_unemployed", "Household unemployed jobseekers", kind="numeric", theme="financial", relevance="medium", circularity="Forms the separate D economic model."),
    spec("primary_earner_status", "Primary-earner status", theme="financial", relevance="high", circularity="Forms the separate D economic model."),
    spec("primary_income_source", "Primary personal income source", theme="financial", relevance="high", circularity="Forms the separate D economic model."),
    spec("edu_funding_source", "Main education funding source", theme="financial", relevance="high", circularity="Forms the separate D economic model."),
]

D_MOTIVATION = [
    spec("study_motivation_type", "Study motivation type", kind="ordered", theme="support", relevance="high", circularity="Forms the separate D motivation model."),
    spec("autonomy_in_study", "Study autonomy", kind="ordered", theme="support", relevance="medium", circularity="Forms the separate D motivation model."),
    spec("competence_in_study_field", "Perceived competence in study field", kind="ordered", theme="support", relevance="high", circularity="Forms the separate D motivation model."),
    spec("relatedness_at_institution", "Institutional belonging", kind="ordered", theme="support", relevance="high", circularity="Forms the separate D motivation model."),
]

E_PROFILERS = COMMON_DEMOGRAPHICS + [
    spec("household_income", "Household income band", kind="ordered", theme="financial", relevance="high"),
    spec("household_size", "Household size", kind="numeric", theme="financial", relevance="medium"),
    spec("primary_earner_status", "Primary-earner status", theme="financial", relevance="high"),
    spec("primary_income_source", "Primary personal income source", theme="financial", relevance="high"),
    spec("income_urgency", "Urgency to start earning", kind="ordered", theme="financial", relevance="high"),
    spec("edu_status", "Recent education status", theme="education", relevance="medium"),
    spec("father_edu", "Father's education", kind="ordered", theme="education", relevance="low"),
    spec("mother_edu", "Mother's education", kind="ordered", theme="education", relevance="low"),
    spec("worked_while_studying", "Worked while studying", theme="employment", relevance="high"),
    spec("work_while_study_type", "Type of work while studying", theme="employment", relevance="high", conditional=True, gateway="worked_while_studying"),
    spec("work_while_study_motivation", "Reason for working while studying", theme="financial", relevance="high", conditional=True, gateway="worked_while_studying"),
    spec("most_important_life_goal", "Most important life goal", theme="aspiration", relevance="high"),
    spec("most_useful_quality_for_job", "Quality considered most useful for a good job", theme="aspiration", relevance="medium"),
    spec("ideal_work_type", "Ideal work type", theme="aspiration", relevance="high"),
    spec("ideal_sector", "Ideal sector", theme="aspiration", relevance="high"),
    spec("job_type_sought", "Type of job sought", theme="aspiration", relevance="high"),
    spec("active_search_4weeks", "Active search in last four weeks", theme="employment", relevance="high"),
    spec("search_method_primary", "Primary recent search method", theme="service", relevance="high", conditional=True, gateway="active_search_4weeks"),
    spec("job_search_duration", "Job-search duration", kind="ordered", theme="barrier", relevance="high"),
    spec("num_applications", "Number of applications", kind="numeric", theme="employment", relevance="medium"),
    spec("num_interviews", "Number of interviews", kind="numeric", theme="employment", relevance="medium"),
    spec("job_acceptance_conditions", "Job-acceptance conditions", kind="ordered", theme="feasibility", relevance="high"),
    spec("has_minimum_wage", "Has a minimum acceptable wage", theme="feasibility", relevance="high"),
    spec("minimum_wage_band", "Minimum acceptable wage band", kind="ordered", theme="feasibility", relevance="high", conditional=True, gateway="has_minimum_wage"),
    spec("ever_refused_job", "Ever refused a job", theme="barrier", relevance="medium"),
    spec("reason_refused_job", "Reason for refusing a job", theme="barrier", relevance="high", conditional=True, gateway="ever_refused_job"),
    spec("employment_service_help", "Employment/careers service help", theme="service", relevance="high"),
    spec("activity_while_searching", "Main activity while searching", theme="employment", relevance="high"),
    spec("willing_to_relocate", "Willingness to relocate", theme="mobility", relevance="high"),
    spec("main_obstacle_finding_work", "Main obstacle to finding work", theme="barrier", relevance="high"),
    spec("training_type_would_help", "Training believed most helpful", theme="training", relevance="high"),
    spec("job_search_self_efficacy", "Job-search confidence", kind="ordered", theme="support", relevance="high"),
    spec("job_search_autonomy_orientation", "Job-search autonomy orientation", kind="ordered", theme="support", relevance="medium"),
    spec("social_support_job_search", "Social support for job search", kind="ordered", theme="support", relevance="high"),
    spec("future_planning_orientation", "Future planning and direction", kind="ordered", theme="support", relevance="high"),
    spec("employment_network_quality", "Useful employment/training contacts", kind="ordered", theme="support", relevance="high"),
    spec("knows_people_in_target_sector", "Knows people in target sector", theme="support", relevance="high"),
    spec("network_helpfulness", "Helpfulness of job-search contacts", theme="support", relevance="high"),
]

F_PROFILERS = COMMON_DEMOGRAPHICS + [
    spec("household_income", "Household income band", kind="ordered", theme="financial", relevance="high"),
    spec("household_size", "Household size", kind="numeric", theme="financial", relevance="medium"),
    spec("primary_earner_status", "Primary-earner status", theme="financial", relevance="high"),
    spec("primary_income_source", "Primary personal income source", theme="financial", relevance="high"),
    spec("edu_status", "Recent education status", theme="education", relevance="medium"),
    spec("worked_while_studying", "Worked while studying", theme="employment", relevance="medium"),
    spec("work_while_study_type", "Type of work while studying", theme="employment", relevance="high", conditional=True, gateway="worked_while_studying = Yes"),
    spec("expected_highest_edu_f", "Expected highest education", kind="ordered", theme="education", relevance="high"),
    spec("most_important_life_goal", "Most important life goal", theme="aspiration", relevance="high"),
    spec("ideal_work_type", "Ideal work type", theme="aspiration", relevance="high"),
    spec("ideal_sector", "Ideal sector", theme="aspiration", relevance="high"),
    spec("employment_status", "Employment arrangement", theme="employment", relevance="high", circularity="Employment arrangement can be associated with contract type."),
    spec("enterprise_location_type", "Workplace/business location", theme="employment", relevance="medium"),
    spec("occupational_class", "Occupational class", theme="employment", relevance="high"),
    spec("has_second_job", "Has a second job", theme="employment", relevance="medium"),
    spec("monthly_income_band", "Monthly work-income band", kind="ordered", theme="financial", relevance="high"),
    spec("usual_hours_per_week", "Usual weekly work hours", kind="numeric", theme="feasibility", relevance="high"),
    spec("actual_hours_last_week", "Actual hours last week", kind="numeric", theme="feasibility", relevance="medium"),
    spec("income_hours_preference", "Preferred income/hours change", theme="feasibility", relevance="high", circularity="This item dominated a rejected D4 clustering and should be interpreted as a direct preference."),
    spec("job_satisfaction", "Overall job satisfaction", kind="ordered", theme="employment", relevance="high"),
    spec("contract_satisfaction", "Contract satisfaction", kind="ordered", theme="employment", relevance="medium", circularity="Closely related to contract type; interpret as an outcome rather than independent validation."),
    spec("hopes_to_change_job", "Hopes to change job", theme="employment", relevance="high"),
    spec("reason_wants_to_change", "Main reason for wanting to change job", theme="barrier", relevance="high", conditional=True, gateway="hopes_to_change_job"),
    spec("willing_to_relocate_f", "Willingness to relocate for better work", theme="mobility", relevance="high"),
    spec("plans_further_edu_f", "Plans further education/training", theme="aspiration", relevance="high"),
    spec("union_member", "Union/worker-association membership", theme="service", relevance="medium"),
    spec("reason_not_union", "Reason for not joining a union", theme="service", relevance="medium", conditional=True, gateway="union_member"),
    spec("received_training", "Received training for current job", theme="training", relevance="high"),
    spec("training_type_received", "Type of training received", theme="training", relevance="high", conditional=True, gateway="received_training"),
    spec("training_provider", "Training provider", theme="training", relevance="high", conditional=True, gateway="received_training"),
    spec("training_duration", "Training duration", kind="ordered", theme="training", relevance="medium", conditional=True, gateway="received_training"),
    spec("who_paid_training", "Who paid for training", theme="financial", relevance="high", conditional=True, gateway="received_training"),
    spec("qualification_job_match", "Qualification/job match", kind="ordered", theme="education", relevance="high"),
    spec("past_edu_useful_for_job", "Usefulness of past education for job", theme="education", relevance="high"),
    spec("professional_network_growth", "Professional-network growth", kind="ordered", theme="support", relevance="high"),
    spec("work_autonomy", "Work autonomy", kind="ordered", theme="support", relevance="medium"),
    spec("competence_growth_at_work", "Competence and skills growth at work", kind="ordered", theme="training", relevance="high"),
    spec("relatedness_at_work", "Workplace connection and support", kind="ordered", theme="support", relevance="medium"),
    spec("work_motivation_type", "Primary work motivation", kind="ordered", theme="support", relevance="medium"),
    spec("registered_as_jobseeker", "Registered as a jobseeker before hire", theme="service", relevance="medium"),
    spec("employment_service_help_f", "Employment-service help before hire", theme="service", relevance="high"),
    spec("jobs_applied_before_hire", "Applications before current job", kind="numeric", theme="employment", relevance="medium"),
    spec("interviews_before_hire", "Interviews before current job", kind="numeric", theme="employment", relevance="medium"),
    spec("search_duration_before_hire", "Search duration before current job", kind="ordered", theme="barrier", relevance="medium"),
    spec("obstacle_finding_this_job", "Main obstacle finding current job", theme="barrier", relevance="high"),
    spec("job_recruitment_method", "Recruitment method", theme="service", relevance="high", conditional=True, gateway="employment_status", circularity="Wage-worker subgroup only."),
    spec("enterprise_size", "Enterprise size", kind="ordered", theme="employment", relevance="medium", conditional=True, gateway="employment_status", circularity="Wage-worker subgroup only."),
    spec("gender_equality_at_work", "Gender equality at work", theme="equity", relevance="medium", conditional=True, gateway="employment_status", circularity="Wage-worker subgroup only."),
    spec("business_profitability", "Business profitability", kind="ordered", theme="financial", relevance="high", conditional=True, gateway="employment_status", circularity="Self-employed subgroup only; n=11."),
    spec("main_business_obstacle", "Main business obstacle", theme="barrier", relevance="high", conditional=True, gateway="employment_status", circularity="Self-employed subgroup only; n=11."),
]

G_GENERAL = COMMON_DEMOGRAPHICS + [
    spec("father_edu", "Father's education", kind="ordered", theme="education", relevance="low"),
    spec("mother_edu", "Mother's education", kind="ordered", theme="education", relevance="low"),
    spec("edu_status", "Recent education status", theme="education", relevance="medium"),
    spec("worked_while_studying", "Worked while studying", theme="employment", relevance="high"),
    spec("work_while_study_type", "Type of work while studying", theme="employment", relevance="high", conditional=True, gateway="worked_while_studying"),
    spec("work_while_study_motivation", "Reason for working while studying", theme="financial", relevance="high", conditional=True, gateway="worked_while_studying"),
    spec("edu_min_for_decent_job", "Minimum education believed necessary for a decent job", kind="ordered", theme="education", relevance="medium"),
    spec("most_important_life_goal", "Most important life goal", theme="aspiration", relevance="high"),
    spec("most_useful_quality_for_job", "Quality considered most useful for a good job", theme="aspiration", relevance="medium"),
    spec("ideal_work_type", "Ideal work type", theme="aspiration", relevance="high"),
    spec("ideal_sector", "Ideal sector", theme="aspiration", relevance="high"),
    spec("time_use_decision_maker", "Who mainly determines daily time use", theme="feasibility", relevance="high"),
    spec("future_orientation_g", "Future orientation", kind="ordered", theme="support", relevance="high"),
    spec("employment_self_efficacy", "Employment/training self-efficacy", kind="ordered", theme="support", relevance="high"),
    spec("support_for_engagement", "Expected support for work/study engagement", kind="ordered", theme="support", relevance="high"),
    spec("employment_network_quality", "Useful employment/training contacts", kind="ordered", theme="support", relevance="high"),
    spec("labour_market_connection", "Connection to labour-market information/opportunities", kind="ordered", theme="support", relevance="high"),
    spec("expected_highest_edu_g", "Desired highest education", kind="ordered", theme="education", relevance="high", conditional=True, gateway="wants_further_edu"),
]

G_ECONOMIC = [
    spec("household_income", "Household income band", kind="ordered", theme="financial", relevance="high", circularity="Forms the separate G economic-role model."),
    spec("household_size", "Household size", kind="numeric", theme="financial", relevance="medium", circularity="Forms the separate G economic-role model."),
    spec("household_workers", "Household workers", kind="numeric", theme="financial", relevance="medium", circularity="Forms the separate G economic-role model."),
    spec("household_unemployed", "Household unemployed jobseekers", kind="numeric", theme="financial", relevance="medium", circularity="Forms the separate G economic-role model."),
    spec("primary_earner_status", "Primary-earner status", theme="financial", relevance="high", circularity="Forms the separate G economic-role model."),
    spec("primary_income_source", "Primary personal income source", theme="financial", relevance="high", circularity="Forms the separate G economic-role model."),
]

G_ENGAGEMENT = [
    spec("reason_not_in_labour_force", "Reason not currently in labour force", theme="barrier", relevance="high", circularity="Forms the separate G engagement model."),
    spec("ever_worked", "Ever worked for pay/profit", theme="employment", relevance="high", circularity="Forms the separate G engagement model."),
    spec("wants_to_work_future", "Wants to work in future", theme="aspiration", relevance="high", circularity="Forms the separate G engagement model."),
    spec("wants_further_edu", "Wants further education/training", theme="aspiration", relevance="high", circularity="Forms the separate G engagement model."),
]

MODELS = {
    "D_motivation": {
        "branch": "D",
        "dimension": "Motivation and study experience",
        "assignment": OUT / "branch_d_motivation_consensus/motivation_consensus_assignments.csv",
        "profile_col": "consensus_profile_id",
        "label_col": "consensus_profile_label",
        "expected_sizes": {1: 47, 2: 75, 3: 69},
        "exclude": {"study_motivation_type", "autonomy_in_study", "competence_in_study_field", "relatedness_at_institution"},
        "derived_exclude": {"study_motivation_type__code", "autonomy_in_study__code", "competence_in_study_field__code", "relatedness_at_institution__code"},
        "profilers": COMMON_DEMOGRAPHICS + D_ECONOMIC + D_EDUCATION + D_ASPIRATIONS + D_NETWORK,
        "caution": "Four single survey items represent motivation, autonomy, competence and relatedness; uncertain memberships must remain visible.",
    },
    "D_economic": {
        "branch": "D",
        "dimension": "Economic resources and household pressure",
        "assignment": OUT / "branch_d_economic_resources_clusters/economic_consensus_assignments.csv",
        "profile_col": "economic_profile_id",
        "label_col": "economic_profile_label",
        "expected_sizes": {1: 44, 2: 147},
        "exclude": {"household_income", "household_size", "household_workers", "household_unemployed", "primary_earner_status", "primary_income_source", "edu_funding_source"},
        "derived_exclude": {"household_income_position", "household_worker_share", "household_unemployed_share", "primary_earner_responsibility", "education_funding_source"},
        "profilers": COMMON_DEMOGRAPHICS + D_EDUCATION + D_MOTIVATION + D_ASPIRATIONS + D_NETWORK,
        "caution": "Income refusals and household-count inconsistencies were handled pairwise in clustering; profiles indicate relative position, not poverty status.",
    },
    "D_educational_cultural": {
        "branch": "D",
        "dimension": "Educational trajectory and cultural capital",
        "assignment": OUT / "branch_d_educational_cultural_capital_clusters/educational_cultural_consensus_assignments.csv",
        "profile_col": "educational_cultural_profile_id",
        "label_col": "educational_cultural_profile_label",
        "expected_sizes": {1: 15, 2: 53, 3: 57, 4: 66},
        "exclude": {"father_edu", "mother_edu", "expected_highest_edu_d", "institution_type", "worked_while_studying", "re_entered_education"},
        "derived_exclude": {"father_education", "mother_education", "expected_education"},
        "profilers": COMMON_DEMOGRAPHICS + D_ECONOMIC + D_MOTIVATION + D_ASPIRATIONS + D_NETWORK + [
            spec("edu_funding_source", "Main education funding source", theme="financial", relevance="high"),
            spec("reason_re_entered", "Reason for re-entering education", theme="education", relevance="high", conditional=True, gateway="re_entered_education", circularity="Direct follow-up to a cluster-forming gateway."),
            spec("work_while_study_type", "Type of work while studying", theme="employment", relevance="high", conditional=True, gateway="worked_while_studying", circularity="Direct follow-up to a cluster-forming gateway."),
            spec("work_while_study_motivation", "Reason for working while studying", theme="financial", relevance="high", conditional=True, gateway="worked_while_studying", circularity="Direct follow-up to a cluster-forming gateway."),
        ],
        "caution": "Parental education is a proxy only; work-experience and re-entry follow-ups are structurally linked to variables defining these profiles.",
    },
    "E_learning_orientation": {
        "branch": "E",
        "dimension": "Education usefulness and further-learning orientation",
        "assignment": OUT / "branch_e_all_dimensions_clusters/e4/consensus_assignments.csv",
        "profile_col": "profile_id",
        "label_col": "profile_label",
        "expected_sizes": {1: 18, 2: 16},
        "exclude": {"past_edu_usefulness", "plans_further_edu", "expected_highest_edu_e"},
        "derived_exclude": set(),
        "profilers": E_PROFILERS,
        "caution": "Branch E has only 34 respondents and the retained solution depends strongly on perceived past-education usefulness.",
    },
    "F_employment_security": {
        "branch": "F",
        "dimension": "Employment security and formality",
        "assignment": OUT / "branch_f_employment_security_clusters/employment_security_consensus_assignments.csv",
        "profile_col": "employment_security_profile_id",
        "label_col": "employment_security_profile_label",
        "expected_sizes": {1: 29, 2: 52},
        "exclude": {"contract_type", "job_security_12months", "sector_formality"},
        "derived_exclude": set(),
        "profilers": F_PROFILERS,
        "caution": "Employment arrangement and contract satisfaction are related outcomes, not independent validation of the security/formality clusters.",
    },
    "G_economic_role": {
        "branch": "G",
        "dimension": "Household economic role and dependency",
        "assignment": OUT / "branch_g_household_economic_resources_clusters/household_economic_consensus_assignments.csv",
        "profile_col": "economic_profile_id",
        "label_col": "economic_profile_label",
        "expected_sizes": {1: 41, 2: 15},
        "exclude": {"household_income", "household_size", "household_workers", "household_unemployed", "primary_earner_status", "primary_income_source"},
        "derived_exclude": {"household_income_position", "household_worker_share", "household_unemployed_share", "primary_earner_responsibility"},
        "profilers": G_GENERAL + G_ENGAGEMENT,
        "caution": "This is an economic-role/dependency pattern, not a low-versus-high household-resource classification.",
    },
    "G_future_engagement": {
        "branch": "G",
        "dimension": "Future engagement and prior work experience",
        "assignment": OUT / "branch_g_inactivity_pathway_future_engagement_clusters/pathway_engagement_consensus_assignments.csv",
        "profile_col": "pathway_profile_id",
        "label_col": "pathway_profile_label",
        "expected_sizes": {1: 35, 2: 21},
        "exclude": {"reason_not_in_labour_force", "ever_worked", "wants_to_work_future", "wants_further_edu"},
        "derived_exclude": set(),
        "profilers": G_GENERAL + G_ECONOMIC,
        "caution": "Current inactivity reason did not define this partition; desired education is conditional on an intention that forms the clusters.",
    },
}

# Remove accidental duplicate profilers within a model while preserving order.
for model in MODELS.values():
    seen = set()
    unique = []
    for item in model["profilers"]:
        if item["variable"] not in seen:
            unique.append(item)
            seen.add(item["variable"])
    model["profilers"] = unique

## Load data and verify frozen assignments

In [2]:
labels_all = pd.read_csv(LABELS_PATH, low_memory=False)
values_all = pd.read_csv(VALUES_PATH, low_memory=False)

assert len(labels_all) == 392 and labels_all["respondent_id"].nunique() == 392
assert len(values_all) == 392 and values_all["respondent_id"].nunique() == 392
assert set(labels_all["respondent_id"]) == set(values_all["respondent_id"])

strict_masks = {
    branch: values_all["branch"].eq(branch)
    & values_all["branch_routing_source"].eq(STRICT_ROUTING_SOURCE)
    for branch in EXPECTED_BRANCH_COUNTS
}
strict_masks["F"] &= values_all["employment_status"].notna()

actual_branch_counts = {branch: int(mask.sum()) for branch, mask in strict_masks.items()}
assert actual_branch_counts == EXPECTED_BRANCH_COUNTS, (actual_branch_counts, EXPECTED_BRANCH_COUNTS)

cohorts = {}
assignment_audit = []
branch_id_sets = {}

for model_id, model in MODELS.items():
    assignment = pd.read_csv(model["assignment"], low_memory=False)
    assert assignment["respondent_id"].notna().all()
    assert assignment["respondent_id"].is_unique

    branch = model["branch"]
    strict_ids = set(values_all.loc[strict_masks[branch], "respondent_id"])
    assignment_ids = set(assignment["respondent_id"])
    assert assignment_ids == strict_ids, f"ID mismatch for {model_id}"

    observed_sizes = (
        assignment[model["profile_col"]].astype(int).value_counts().sort_index().to_dict()
    )
    assert observed_sizes == model["expected_sizes"], (model_id, observed_sizes)

    if branch in branch_id_sets:
        assert branch_id_sets[branch] == assignment_ids
    else:
        branch_id_sets[branch] = assignment_ids

    profiler_vars = {item["variable"] for item in model["profilers"]}
    forbidden = model["exclude"] | model["derived_exclude"]
    assert profiler_vars.isdisjoint(forbidden), f"Forbidden profiler in {model_id}: {profiler_vars & forbidden}"
    assert profiler_vars.issubset(labels_all.columns), f"Missing label variable in {model_id}"
    assert profiler_vars.issubset(values_all.columns), f"Missing value variable in {model_id}"

    assignment_keep = assignment[[
        "respondent_id",
        model["profile_col"],
        model["label_col"],
    ]].rename(columns={
        model["profile_col"]: "profile_id",
        model["label_col"]: "profile_label",
    })

    label_cohort = labels_all.loc[strict_masks[branch], ["respondent_id"] + sorted(profiler_vars)].copy()
    value_cohort = values_all.loc[strict_masks[branch], ["respondent_id"] + sorted(profiler_vars)].copy()
    merged = assignment_keep.merge(label_cohort, on="respondent_id", validate="one_to_one")
    value_renamed = value_cohort.rename(columns={v: f"{v}__value" for v in profiler_vars})
    merged = merged.merge(value_renamed, on="respondent_id", validate="one_to_one")
    assert len(merged) == EXPECTED_BRANCH_COUNTS[branch]
    cohorts[model_id] = merged

    assignment_audit.append({
        "model_id": model_id,
        "branch": branch,
        "dimension": model["dimension"],
        "records": len(merged),
        "unique_ids": merged["respondent_id"].nunique(),
        "profile_sizes": json.dumps(observed_sizes),
        "id_set_matches_strict_cohort": True,
        "excluded_variables": ", ".join(sorted(forbidden)),
    })

assert len(set().union(*branch_id_sets.values())) == 362
assignment_audit_df = pd.DataFrame(assignment_audit)
print("Strict cohorts verified:", actual_branch_counts, "Total unique respondents: 362")
display(assignment_audit_df)

Strict cohorts verified: {'D': 191, 'E': 34, 'F': 81, 'G': 56} Total unique respondents: 362


,model_id,branch,dimension,records,unique_ids,profile_sizes,id_set_matches_strict_cohort,excluded_variables
0,D_motivation,D,Motivation and study experience,191,191,"{""1"": 47, ""2"": 75, ""3"": 69}",True,"autonomy_in_study, autonomy_in_study__code, competence_in_study_field, competence_in_study_field__code, relatedness_at_institution, relatedness_at_institution__code, study_moti..."
1,D_economic,D,Economic resources and household pressure,191,191,"{""1"": 44, ""2"": 147}",True,"edu_funding_source, education_funding_source, household_income, household_income_position, household_size, household_unemployed, household_unemployed_share, household_worker_sh..."
2,D_educational_cultural,D,Educational trajectory and cultural capital,191,191,"{""1"": 15, ""2"": 53, ""3"": 57, ""4"": 66}",True,"expected_education, expected_highest_edu_d, father_edu, father_education, institution_type, mother_edu, mother_education, re_entered_education, worked_while_studying"
3,E_learning_orientation,E,Education usefulness and further-learning orientation,34,34,"{""1"": 18, ""2"": 16}",True,"expected_highest_edu_e, past_edu_usefulness, plans_further_edu"
4,F_employment_security,F,Employment security and formality,81,81,"{""1"": 29, ""2"": 52}",True,"contract_type, job_security_12months, sector_formality"
5,G_economic_role,G,Household economic role and dependency,56,56,"{""1"": 41, ""2"": 15}",True,"household_income, household_income_position, household_size, household_unemployed, household_unemployed_share, household_worker_share, household_workers, primary_earner_respons..."
6,G_future_engagement,G,Future engagement and prior work experience,56,56,"{""1"": 35, ""2"": 21}",True,"ever_worked, reason_not_in_labour_force, wants_further_edu, wants_to_work_future"


## Comparison functions

Categorical variables use counts, within-profile percentages, Cramér's V and a
chi-square/Fisher supporting test. Ordered and numeric variables use medians and IQRs,
with Mann–Whitney or Kruskal–Wallis tests and rank-biserial or epsilon-squared effect
sizes. False-discovery-rate adjustment is performed within each retained dimension.

In [3]:
NON_SUBSTANTIVE_PATTERN = re.compile(
    r"^(prefer not to say|prefer not to answer|unknown|don't know|do not know|refused)$",
    flags=re.IGNORECASE,
)


def valid_label_mask(series):
    text = series.astype("string").str.strip()
    return series.notna() & ~text.str.match(NON_SUBSTANTIVE_PATTERN, na=False)


def benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    out = np.full(len(p), np.nan)
    valid = np.isfinite(p)
    if not valid.any():
        return out
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    restored = np.empty_like(adjusted)
    restored[order] = adjusted
    out[np.where(valid)[0]] = restored
    return out


def cramer_v(table):
    arr = np.asarray(table, dtype=float)
    n = arr.sum()
    if n == 0 or min(arr.shape) < 2:
        return 0.0, np.nan, True, 1.0
    try:
        chi2, p, _, expected = stats.chi2_contingency(arr, correction=False)
    except ValueError:
        return 0.0, np.nan, True, 1.0
    phi2 = chi2 / n
    r, k = arr.shape
    phi2_corr = max(0.0, phi2 - ((k - 1) * (r - 1)) / max(n - 1, 1))
    r_corr = r - ((r - 1) ** 2) / max(n - 1, 1)
    k_corr = k - ((k - 1) ** 2) / max(n - 1, 1)
    denom = min(k_corr - 1, r_corr - 1)
    value = math.sqrt(phi2_corr / denom) if denom > 0 else 0.0
    sparse_share = float((expected < 5).mean())
    sparse = bool(sparse_share > 0.20 or (expected < 1).any())
    if arr.shape == (2, 2) and sparse:
        _, p = stats.fisher_exact(arr)
    return float(value), float(p), sparse, sparse_share


def categorical_description(frame, variable):
    pieces = []
    for profile_id, group in frame.groupby("profile_id", sort=True):
        valid = group.loc[valid_label_mask(group[variable]), variable].astype(str)
        denom = len(valid)
        counts = valid.value_counts(dropna=False)
        cats = "; ".join(
            f"{category}: {count} ({100 * count / denom:.1f}%)"
            for category, count in counts.items()
        ) if denom else "no eligible responses"
        pieces.append(f"Profile {int(profile_id)} eligible n={denom}: {cats}")
    return " || ".join(pieces)


def categorical_contrasts(frame, variable):
    valid = frame.loc[valid_label_mask(frame[variable]), ["profile_id", variable]].copy()
    valid[variable] = valid[variable].astype(str)
    contrasts = {}
    for profile_id in sorted(frame["profile_id"].unique()):
        this = valid.loc[valid["profile_id"].eq(profile_id), variable]
        other = valid.loc[~valid["profile_id"].eq(profile_id), variable]
        if len(this) == 0 or len(other) == 0:
            contrasts[int(profile_id)] = "insufficient eligible responses"
            continue
        categories = sorted(set(this) | set(other))
        candidates = []
        for category in categories:
            p_this = float(this.eq(category).mean())
            p_other = float(other.eq(category).mean())
            candidates.append((abs(p_this - p_other), p_this - p_other, category, p_this, p_other, int(this.eq(category).sum()), len(this)))
        _, difference, category, p_this, p_other, count, denom = max(candidates)
        direction = "more often" if difference >= 0 else "less often"
        contrasts[int(profile_id)] = (
            f"{direction} reported ‘{category}’ ({count}/{denom}, {100*p_this:.1f}% vs "
            f"{100*p_other:.1f}% in other profile(s))"
        )
    return contrasts


def ordered_or_numeric_result(frame, item):
    variable = item["variable"]
    raw = pd.to_numeric(frame[f"{variable}__value"], errors="coerce")
    if item["kind"] == "ordered":
        raw = raw.mask(raw >= 90)
    valid = raw.notna() & frame["profile_id"].notna()
    working = pd.DataFrame({"profile_id": frame.loc[valid, "profile_id"].astype(int), "value": raw.loc[valid]})
    groups = [g["value"].to_numpy(dtype=float) for _, g in working.groupby("profile_id", sort=True)]
    group_ids = [int(k) for k, _ in working.groupby("profile_id", sort=True)]
    if len(groups) < 2 or any(len(g) == 0 for g in groups):
        effect, p, effect_name = 0.0, np.nan, "unavailable"
    elif len(groups) == 2:
        test = stats.mannwhitneyu(groups[0], groups[1], alternative="two-sided")
        effect = abs(1 - 2 * float(test.statistic) / (len(groups[0]) * len(groups[1])))
        p = float(test.pvalue)
        effect_name = "absolute rank-biserial correlation"
    else:
        test = stats.kruskal(*groups)
        h = float(test.statistic)
        effect = max(0.0, (h - len(groups) + 1) / max(len(working) - len(groups), 1))
        p = float(test.pvalue)
        effect_name = "epsilon-squared"

    descriptions = []
    contrasts = {}
    for profile_id in sorted(frame["profile_id"].unique()):
        values = working.loc[working["profile_id"].eq(profile_id), "value"]
        if len(values):
            q1, med, q3 = values.quantile([0.25, 0.5, 0.75])
            descriptions.append(f"Profile {int(profile_id)} n={len(values)}: median={med:.2f}, IQR={q1:.2f}–{q3:.2f}")
            other = working.loc[~working["profile_id"].eq(profile_id), "value"]
            if len(other):
                other_med = float(other.median())
                direction = "higher" if med > other_med else "lower" if med < other_med else "the same"
                contrasts[int(profile_id)] = f"had {direction} median {item['variable_label'].lower()} ({med:.2f} vs {other_med:.2f})"
            else:
                contrasts[int(profile_id)] = "insufficient comparison responses"
        else:
            descriptions.append(f"Profile {int(profile_id)} n=0")
            contrasts[int(profile_id)] = "insufficient eligible responses"

    # For ordered variables, a readable category contrast is more interpretable than a code alone.
    if item["kind"] == "ordered":
        readable = frame.copy()
        readable.loc[~valid, variable] = np.nan
        contrasts = categorical_contrasts(readable, variable)

    profile_eligible = working.groupby("profile_id").size().reindex(sorted(frame["profile_id"].unique()), fill_value=0)
    return {
        "eligible_n": int(valid.sum()),
        "coverage_pct": float(100 * valid.mean()),
        "effect_size_name": effect_name,
        "effect_size": float(effect),
        "p_value": p,
        "sparse_cells": False,
        "sparse_expected_share": np.nan,
        "minimum_profile_eligible_n": int(profile_eligible.min()),
        "profile_descriptives": " || ".join(descriptions),
        "contrasts": contrasts,
    }


def categorical_result(frame, item):
    variable = item["variable"]
    valid = valid_label_mask(frame[variable]) & frame["profile_id"].notna()
    working = frame.loc[valid, ["profile_id", variable]].copy()
    table = pd.crosstab(working["profile_id"], working[variable])
    effect, p, sparse, sparse_share = cramer_v(table)
    expected_profiles = sorted(frame["profile_id"].unique())
    profile_eligible = working.groupby("profile_id").size().reindex(expected_profiles, fill_value=0)
    return {
        "eligible_n": int(valid.sum()),
        "coverage_pct": float(100 * valid.mean()),
        "effect_size_name": "bias-corrected Cramér's V",
        "effect_size": effect,
        "p_value": p,
        "sparse_cells": sparse,
        "sparse_expected_share": sparse_share,
        "minimum_profile_eligible_n": int(profile_eligible.min()),
        "profile_descriptives": categorical_description(frame, variable),
        "contrasts": categorical_contrasts(frame, variable),
    }


def effect_threshold(effect_name):
    if "Cramér" in effect_name:
        return 0.15
    if "rank-biserial" in effect_name:
        return 0.20
    if "epsilon" in effect_name:
        return 0.03
    return 1.0


RELEVANCE_WEIGHT = {"high": 1.0, "medium": 0.80, "low": 0.55}

## Run the seven frozen post-cluster comparisons

In [4]:
comparison_rows = []
contrast_lookup = {}

for model_id, model in MODELS.items():
    frame = cohorts[model_id]
    for item in model["profilers"]:
        if item["kind"] in {"ordered", "numeric"}:
            result = ordered_or_numeric_result(frame, item)
        else:
            result = categorical_result(frame, item)

        small_sample = (
            result["eligible_n"] < 30
            or result["minimum_profile_eligible_n"] < 10
            or bool(item["conditional"])
        )
        coverage_factor = min(1.0, result["coverage_pct"] / 80.0)
        conditional_factor = 0.80 if item["conditional"] else 1.0
        circular_factor = 0.65 if item["circularity"] else 1.0
        sparse_factor = 0.65 if result["sparse_cells"] else 1.0
        importance = (
            result["effect_size"]
            * coverage_factor
            * RELEVANCE_WEIGHT[item["programme_relevance"]]
            * conditional_factor
            * circular_factor
            * sparse_factor
        )
        reporting_threshold = effect_threshold(result["effect_size_name"])
        # Sparse categorical tables can produce deceptively large apparent differences.
        # Require a clearly larger effect before allowing them into concise profile text.
        if result["sparse_cells"]:
            reporting_threshold = max(reporting_threshold, 0.25)
        meaningful = (
            result["effect_size"] >= reporting_threshold
            and result["coverage_pct"] >= 40
            and result["minimum_profile_eligible_n"] >= 5
            and item["programme_relevance"] != "low"
        )
        comparison_rows.append({
            "model_id": model_id,
            "branch": model["branch"],
            "dimension": model["dimension"],
            "variable": item["variable"],
            "variable_label": item["variable_label"],
            "variable_type": item["kind"],
            "theme": item["theme"],
            "programme_relevance": item["programme_relevance"],
            "conditional": item["conditional"],
            "gateway": item["gateway"],
            "circularity_note": item["circularity"],
            "total_model_n": len(frame),
            **{k: v for k, v in result.items() if k != "contrasts"},
            "small_sample_flag": small_sample,
            "importance_score": importance,
            "passes_effect_coverage_screen": meaningful,
        })
        contrast_lookup[(model_id, item["variable"])] = result["contrasts"]

comparison = pd.DataFrame(comparison_rows)
approved_f_work_mask = (
    comparison["model_id"].eq("F_employment_security")
    & comparison["variable"].eq("work_while_study_type")
)
assert int(approved_f_work_mask.sum()) == 1
# Retain all seven audited categories, including zero cells, in the authoritative row.
comparison.loc[approved_f_work_mask, "profile_descriptives"] = (
    "Profile 1 eligible n=22: Informal or casual work: 0 (0.0%); Internship or apprenticeship in a private company: 17 (77.3%); "
    "Internship or apprenticeship in the public sector: 1 (4.5%); Paid work in a company (not internship): 3 (13.6%); "
    "Paid work in the public sector (not internship): 0 (0.0%); Volunteer or community work (unpaid): 0 (0.0%); "
    "Work in a family business or farm: 1 (4.5%) || Profile 2 eligible n=41: Informal or casual work: 1 (2.4%); "
    "Internship or apprenticeship in a private company: 12 (29.3%); Internship or apprenticeship in the public sector: 0 (0.0%); "
    "Paid work in a company (not internship): 23 (56.1%); Paid work in the public sector (not internship): 4 (9.8%); "
    "Volunteer or community work (unpaid): 1 (2.4%); Work in a family business or farm: 0 (0.0%)"
)
comparison["p_adjusted_bh"] = np.nan
for model_id, idx in comparison.groupby("model_id").groups.items():
    # Preserve the original within-model BH family for the frozen findings. The approved
    # Branch F work-type row stores the separately audited full-profile sensitivity BH.
    if model_id == "F_employment_security":
        primary_idx = comparison.loc[idx].index[
            comparison.loc[idx, "variable"].ne("work_while_study_type")
        ]
        comparison.loc[primary_idx, "p_adjusted_bh"] = benjamini_hochberg(
            comparison.loc[primary_idx, "p_value"]
        )
        secondary_idx = comparison.loc[idx].index[
            comparison.loc[idx, "variable"].eq("work_while_study_type")
        ]
        assert len(primary_idx) == 57 and len(secondary_idx) == 1
        comparison.loc[secondary_idx, "p_adjusted_bh"] = 0.049183888149129905
    else:
        comparison.loc[idx, "p_adjusted_bh"] = benjamini_hochberg(comparison.loc[idx, "p_value"])

# Supporting p-values do not control selection; this column is reported for transparency.
comparison["supporting_fdr_lt_0_05"] = comparison["p_adjusted_bh"].lt(0.05)
comparison = comparison.sort_values(["model_id", "importance_score"], ascending=[True, False]).reset_index(drop=True)
comparison.to_csv(COMPARISON_OUT, index=False)

print(f"Wrote {COMPARISON_OUT.relative_to(ROOT)} with {len(comparison)} model-variable comparisons")
display(comparison[[
    "model_id", "variable_label", "coverage_pct", "effect_size_name", "effect_size",
    "p_adjusted_bh", "importance_score", "passes_effect_coverage_screen", "conditional",
    "sparse_cells"
]].groupby("model_id", group_keys=False).head(8))

Wrote outputs/final_cluster_profile_comparison_summary.csv with 269 model-variable comparisons


,model_id,variable_label,coverage_pct,effect_size_name,effect_size,p_adjusted_bh,importance_score,passes_effect_coverage_screen,conditional,sparse_cells
0,D_economic,Ideal sector,100.000000,bias-corrected Cramér's V,0.380732,9.369311e-04,0.247476,True,False,True
1,D_economic,Current or preferred field of study,97.905759,bias-corrected Cramér's V,0.377931,3.569755e-04,0.245655,True,False,True
2,D_economic,Current institution type,100.000000,bias-corrected Cramér's V,0.566465,2.371697e-11,0.239332,True,False,True
3,D_economic,How current programme was accessed,100.000000,bias-corrected Cramér's V,0.369558,8.441625e-05,0.192170,True,False,True
4,D_economic,Main job-search method while studying,58.115183,bias-corrected Cramér's V,0.299848,1.122420e-01,0.113267,True,True,True
5,D_economic,Plans after current education,100.000000,bias-corrected Cramér's V,0.142448,2.841056e-01,0.092591,False,False,True
6,D_economic,Has children,100.000000,bias-corrected Cramér's V,0.173924,1.772610e-01,0.090441,False,False,True
7,D_economic,Ideal work type,100.000000,bias-corrected Cramér's V,0.131382,3.905903e-01,0.085398,False,False,True
34,D_educational_cultural,Already searching for work/business,100.000000,bias-corrected Cramér's V,0.263131,4.111770e-03,0.263131,True,False,False
35,D_educational_cultural,Primary personal income source,100.000000,bias-corrected Cramér's V,0.364835,1.285032e-10,0.154143,True,False,True


## Build concise evidence summaries for all 17 profiles

In [5]:
THEME_GROUPS = {
    "barriers_and_support_needs": {"barrier", "support", "financial"},
    "aspirations_and_preferences": {"aspiration", "mobility", "feasibility"},
    "education_training_employment_characteristics": {"education", "training", "employment", "service"},
}


def evidence_statement(model_id, profile_id, row):
    contrast = contrast_lookup[(model_id, row["variable"])].get(int(profile_id), "insufficient comparison responses")
    caveat = []
    if row["conditional"]:
        caveat.append(f"conditional eligible n={int(row['eligible_n'])}")
    if row["sparse_cells"]:
        caveat.append("sparse table")
    if row["circularity_note"]:
        caveat.append("cross-dimensional/related variable")
    suffix = f" [{'; '.join(caveat)}]" if caveat else ""
    return f"{row['variable_label']}: {contrast}{suffix}"


profile_rows = []
plain_dimension_summaries = []

for model_id, model in MODELS.items():
    frame = cohorts[model_id]
    model_results = comparison.loc[comparison["model_id"].eq(model_id)].copy()
    # The approved F work-type result is a secondary conditional finding. Keep it out
    # of the automated primary ranking so no previously frozen F finding is displaced.
    primary_results = model_results.loc[
        ~((model_results["model_id"].eq("F_employment_security"))
          & model_results["variable"].eq("work_while_study_type"))
    ].copy()
    screened = primary_results.loc[primary_results["passes_effect_coverage_screen"]].copy()
    if screened.empty:
        screened = primary_results.loc[
            primary_results["coverage_pct"].ge(60)
            & primary_results["programme_relevance"].isin(["high", "medium"])
        ].head(3)
    else:
        screened = screened.head(10)

    headline_vars = screened.head(3)["variable_label"].tolist()
    if headline_vars:
        plain_dimension_summaries.append(
            f"{model['branch']} — {model['dimension']}: strongest additional differences concerned "
            + ", ".join(headline_vars)
            + "."
        )
    else:
        plain_dimension_summaries.append(
            f"{model['branch']} — {model['dimension']}: no selected non-clustering variable showed a clear large difference."
        )

    profile_info = frame[["profile_id", "profile_label"]].drop_duplicates().sort_values("profile_id")
    for _, profile in profile_info.iterrows():
        profile_id = int(profile["profile_id"])
        cluster_size = int(frame["profile_id"].eq(profile_id).sum())
        statements = [evidence_statement(model_id, profile_id, row) for _, row in screened.head(5).iterrows()]
        strongest = " | ".join(statements) if statements else "No large additional difference met the reporting screen."

        themed = {}
        for output_col, themes in THEME_GROUPS.items():
            candidates = screened.loc[screened["theme"].isin(themes)].head(3)
            themed[output_col] = (
                " | ".join(evidence_statement(model_id, profile_id, row) for _, row in candidates.iterrows())
                if len(candidates)
                else "No clear additional difference met the reporting screen."
            )

        programme_candidates = screened.loc[screened["programme_relevance"].eq("high")].head(4)
        programme_findings = (
            " | ".join(evidence_statement(model_id, profile_id, row) for _, row in programme_candidates.iterrows())
            if len(programme_candidates)
            else "No high-priority programme-relevant difference met the reporting screen."
        )

        cautions = [model["caution"]]
        if model_id == "F_employment_security":
            if profile_id == 1:
                secondary_text = (
                    "Among respondents in this profile who worked while studying, prior work experience "
                    "was predominantly through private-company internships/apprenticeships (17/22, 77.3%) "
                    "[conditional secondary finding; severe sparse table]"
                )
            else:
                secondary_text = (
                    "Among respondents in this profile who worked while studying, ordinary paid private-company "
                    "work outside internships was more common (23/41, 56.1%), while private-company "
                    "internships/apprenticeships accounted for 12/41 (29.3%) "
                    "[conditional secondary finding; severe sparse table]"
                )
            if themed["education_training_employment_characteristics"].startswith("No clear"):
                themed["education_training_employment_characteristics"] = secondary_text
            else:
                themed["education_training_employment_characteristics"] += " | " + secondary_text
        if cluster_size < 20:
            cautions.append(f"Small profile (n={cluster_size}).")
        if screened["conditional"].any():
            cautions.append("Some reported findings use eligible-subgroup denominators.")
        if screened["sparse_cells"].any():
            cautions.append("Some categorical comparisons contain sparse cells.")
        if model_id == "F_employment_security":
            cautions.append(
                "The work-type association applies only to the 63 respondents who worked while studying "
                "and has severe sparse cells (71.4% of expected cells below 5); it is non-causal and "
                "does not indicate that internships were unpaid."
            )
        if not screened["supporting_fdr_lt_0_05"].any():
            cautions.append("None of the selected comparisons remained below FDR 0.05; effect sizes are descriptive evidence.")

        profile_rows.append({
            "model_id": model_id,
            "branch": model["branch"],
            "dimension": model["dimension"],
            "profile_id": profile_id,
            "profile_label": profile["profile_label"],
            "cluster_size": cluster_size,
            "strongest_non_clustering_characteristics": strongest,
            **themed,
            "programme_relevant_findings": programme_findings,
            "cautions": " ".join(cautions),
        })

profiles = pd.DataFrame(profile_rows).sort_values(["branch", "model_id", "profile_id"]).reset_index(drop=True)
assert len(profiles) == 17
profiles.to_csv(PROFILE_OUT, index=False)
print(f"Wrote {PROFILE_OUT.relative_to(ROOT)} with {len(profiles)} retained profiles")
display(profiles[["branch", "dimension", "profile_id", "profile_label", "cluster_size"]])

Wrote outputs/final_cluster_programme_relevant_profiles.csv with 17 retained profiles


,branch,dimension,profile_id,profile_label,cluster_size
0,D,Economic resources and household pressure,1,Economic profile 1: relatively lower household economic position,44
1,D,Economic resources and household pressure,2,Economic profile 2: relatively higher household economic position,147
2,D,Educational trajectory and cultural capital,1,"Re-entered students, often with degree-educated parents",15
3,D,Educational trajectory and cultural capital,2,State-institution students without work exposure,53
4,D,Educational trajectory and cultural capital,3,Private-institution students without work exposure,57
5,D,Educational trajectory and cultural capital,4,"Work-experienced students, mainly in private institutions",66
6,D,Motivation and study experience,1,"Consensus motivation profile 1: externally or uncertainly motivated, with moderate study support",47
7,D,Motivation and study experience,2,Consensus motivation profile 2: goal-aligned motivation with mixed study confidence and support,75
8,D,Motivation and study experience,3,"Consensus motivation profile 3: highly self-directed, capable and connected",69
9,E,Education usefulness and further-learning orientation,1,High perceived education utility and strong learning ambition,18


## Generate the plain-language Markdown report

In [6]:
def md_escape(text):
    return str(text).replace("|", "\\|").replace("\n", " ")


report = []
report.extend([
    "# NYSC final post-cluster profiling report",
    "",
    "**Status:** Descriptive post-cluster profiling of the seven retained dimension-specific models  ",
    "**Cohort:** 362 unique respondents — D=191, E=34, F=81, G=56  ",
    "**Important:** These findings describe additional survey characteristics. They are not programme recommendations and do not change any cluster assignment.",
    "",
    "## 1. What this analysis does",
    "",
    "For each retained dimension, this analysis compares its profiles using selected survey questions that did not create that particular clustering. It focuses on needs, barriers, aspirations, education/work experience, mobility, service use and delivery feasibility.",
    "",
    "Counts and within-profile percentages are used for categorical questions. Medians and interquartile ranges are used for ordered and numeric questions. Effect size is the main ranking evidence; p-values are supporting evidence and are adjusted within each model using Benjamini–Hochberg false-discovery-rate control.",
    "",
    "Conditional questions are analysed only among eligible respondents. A blank caused by survey routing is never treated as a No answer.",
    "",
    "## 2. Cohort and assignment verification",
    "",
    "| Branch | Verified respondents | Retained models | Retained profiles |",
    "|---|---:|---:|---:|",
    "| D — In education | 191 | 3 | 9 |",
    "| E — Unemployed/searching | 34 | 1 | 2 |",
    "| F — Employed | 81 | 1 | 2 |",
    "| G — Inactive | 56 | 2 | 4 |",
    "| **Total unique** | **362** | **7** | **17** |",
    "",
])

section_number = 3
for model_id, model in MODELS.items():
    model_profiles = profiles.loc[profiles["model_id"].eq(model_id)]
    top_primary = comparison.loc[
        comparison["model_id"].eq(model_id)
        & comparison["passes_effect_coverage_screen"]
        & ~((comparison["model_id"].eq("F_employment_security"))
            & comparison["variable"].eq("work_while_study_type"))
    ].head(8)
    if model_id == "F_employment_security":
        secondary_top = comparison.loc[
            comparison["model_id"].eq(model_id)
            & comparison["variable"].eq("work_while_study_type")
        ]
        top = pd.concat([top_primary.iloc[:4], secondary_top, top_primary.iloc[4:]])
    else:
        top = top_primary
    report.extend([
        f"## {section_number}. Branch {model['branch']} — {model['dimension']}",
        "",
        f"**Cluster-forming variables excluded:** `{ '`, `'.join(sorted(model['exclude'])) }`.",
        "",
        f"**Model caution:** {model['caution']}",
        "",
    ])
    if len(top):
        report.extend([
            "### Strongest additional comparison variables",
            "",
            "| Variable | Coverage | Effect size | FDR-adjusted p | Programme relevance | Flags |",
            "|---|---:|---:|---:|---|---|",
        ])
        for _, row in top.iterrows():
            flags = []
            if row["conditional"]:
                flags.append("conditional")
            if row["sparse_cells"]:
                if model_id == "F_employment_security" and row["variable"] == "work_while_study_type":
                    flags.append("severe sparse table")
                else:
                    flags.append("sparse")
            if row["circularity_note"]:
                flags.append("related/cross-dimensional")
            p_text = "NA" if pd.isna(row["p_adjusted_bh"]) else f"{row['p_adjusted_bh']:.3f}"
            report.append(
                f"| {md_escape(row['variable_label'])} | {row['coverage_pct']:.1f}% | "
                f"{row['effect_size']:.3f} ({md_escape(row['effect_size_name'])}) | {p_text} | "
                f"{row['programme_relevance']} | {', '.join(flags) if flags else '—'} |"
            )
        report.append("")
    else:
        report.extend([
            "No non-clustering variable met the prespecified effect-size, coverage and minimum-eligible-size screen. This does not mean the profiles are identical; it means no additional difference was strong enough for concise reporting.",
            "",
        ])

    if model_id == "F_employment_security":
        report.extend([
            "### Approved secondary conditional finding",
            "",
            "Among the 63 Branch F respondents who reported working while studying, F-S1 was much more commonly associated with private-company internship/apprenticeship experience (17/22, 77.3%) than F-S2 (12/41, 29.3%). F-S2 was more commonly associated with ordinary paid company work outside internships (23/41, 56.1%) than F-S1 (3/22, 13.6%). This was a strong observed association (bias-corrected Cramér's V = 0.480; raw p = 0.0024; BH-adjusted p across the five targeted checks = 0.0119; sensitivity BH-adjusted p when those checks were appended to the existing F profiling family = 0.0492).",
            "",
            "This comparison is conditional and does not apply to all 81 Branch F respondents. The profiles did not meaningfully differ in whether respondents worked while studying: 22/29 (75.9%) in F-S1 and 41/52 (78.8%) in F-S2. The work-type table was severely sparse, with 71.4% of expected cells below 5, because several categories were rare; those rare categories are not interpreted individually. The substantive comparison is limited to the two dominant categories above. This is an association, not evidence that earlier work type caused later employment security. It also does not mean F-S1 internships were unpaid: paid work-while-studying status was reported by 19/22 (86.4%) in F-S1 and 39/41 (95.1%) in F-S2, a difference that was not meaningful.",
            "",
        ])

    report.append("### Profile summaries")
    report.append("")
    for _, row in model_profiles.iterrows():
        report.extend([
            f"#### Profile {int(row['profile_id'])}: {row['profile_label']} — n={int(row['cluster_size'])}",
            "",
            f"- **Strongest additional characteristics:** {row['strongest_non_clustering_characteristics']}",
            f"- **Barriers/support/financial context:** {row['barriers_and_support_needs']}",
            f"- **Aspirations and preferences:** {row['aspirations_and_preferences']}",
            f"- **Education, training and employment:** {row['education_training_employment_characteristics']}",
            f"- **Programme-relevant evidence:** {row['programme_relevant_findings']}",
            f"- **Caution:** {row['cautions']}",
            "",
        ])
    section_number += 1

report.extend([
    f"## {section_number}. How to read these findings safely",
    "",
    "1. A reported percentage is a within-profile percentage, not a population estimate for all Sri Lankan youth.",
    "2. Effect sizes rank descriptive differences; they do not prove that cluster membership caused the characteristic.",
    "3. FDR-adjusted p-values are supporting evidence only, especially for Branch E and profiles of size 15.",
    "4. Conditional findings use eligible respondents only and show their eligible n in the comparison file.",
    "5. Cross-dimensional variables are marked because they form another retained model. Their relationship is descriptive, not independent validation.",
    "6. High-cardinality variables such as district, sector and obstacle can contain sparse cells and require cautious interpretation.",
    "7. The findings identify possible programme-relevant characteristics; they do not select or recommend a programme.",
    "",
    f"## {section_number + 1}. Reproducible outputs",
    "",
    "- `30_final_cluster_profile_analysis.ipynb`",
    "- `outputs/final_cluster_profile_comparison_summary.csv`",
    "- `outputs/final_cluster_programme_relevant_profiles.csv`",
    "- `final_cluster_profiling_report.md`",
    "",
])

REPORT_OUT.write_text("\n".join(report), encoding="utf-8")
print(f"Wrote {REPORT_OUT.relative_to(ROOT)}")

Wrote final_cluster_profiling_report.md


## Approved Branch F secondary conditional finding

A later targeted audit approved `work_while_study_type` as a secondary/cautious profiling finding for the frozen F employment-security profiles. It is conditional on `worked_while_studying = Yes` and applies to 63 respondents only (F-S1 n=22; F-S2 n=41; structural N/A n=18). The comparison export contains the complete category distribution, and the final checks below verify the saved denominators and statistics without rerunning clustering.

Interpretation is restricted to association: F-S1 more often reported private-company internship/apprenticeship experience, whereas F-S2 more often reported ordinary paid company work outside internships. Severe sparse cells require caution; this does not imply causation or unpaid internships.

In [ ]:
# Final reproducibility checks for the approved targeted Branch F addition.
f_review = cohorts["F_employment_security"].copy()
assert f_review["profile_id"].value_counts().sort_index().to_dict() == {1: 29, 2: 52}

f_eligible = f_review.loc[f_review["worked_while_studying"].eq("Yes")].copy()
assert f_eligible["profile_id"].value_counts().sort_index().to_dict() == {1: 22, 2: 41}
assert len(f_eligible) == 63
assert int((~valid_label_mask(f_review["work_while_study_type"])).sum()) == 18

work_categories = [
    "Informal or casual work",
    "Internship or apprenticeship in a private company",
    "Internship or apprenticeship in the public sector",
    "Paid work in a company (not internship)",
    "Paid work in the public sector (not internship)",
    "Volunteer or community work (unpaid)",
    "Work in a family business or farm",
]
work_table = pd.crosstab(f_eligible["profile_id"], f_eligible["work_while_study_type"]).reindex(
    index=[1, 2], columns=work_categories, fill_value=0
)
expected_work_table = pd.DataFrame(
    [[0, 17, 1, 3, 0, 0, 1], [1, 12, 0, 23, 4, 1, 0]],
    index=pd.Index([1, 2], name="profile_id"),
    columns=pd.Index(work_categories, name="work_while_study_type"),
)
pd.testing.assert_frame_equal(work_table, expected_work_table)

work_effect, work_p, work_sparse, work_sparse_share = cramer_v(work_table)
assert np.isclose(work_effect, 0.4798545525713314)
assert np.isclose(work_p, 0.0023798655556030598)
assert work_sparse and np.isclose(work_sparse_share, 0.7142857142857143)

targeted_work_type_bh = 0.0118993277780153
full_profile_sensitivity_bh = 0.049183888149129905
stored_work_row = comparison.loc[
    comparison["model_id"].eq("F_employment_security")
    & comparison["variable"].eq("work_while_study_type")
].squeeze()
assert len(comparison.loc[comparison["model_id"].eq("F_employment_security")]) == 58
assert int(stored_work_row["eligible_n"]) == 63
assert int(stored_work_row["minimum_profile_eligible_n"]) == 22
assert np.isclose(stored_work_row["effect_size"], work_effect)
assert np.isclose(stored_work_row["p_value"], work_p)
assert np.isclose(stored_work_row["p_adjusted_bh"], full_profile_sensitivity_bh)
assert bool(stored_work_row["sparse_cells"])

paid_check = f_eligible[["respondent_id", "profile_id"]].merge(
    labels_all[["respondent_id", "work_while_study_paid"]],
    on="respondent_id", validate="one_to_one"
)
paid_table = pd.crosstab(paid_check["profile_id"], paid_check["work_while_study_paid"])
assert paid_table.loc[1].to_dict() == {"Paid": 19, "Unpaid": 3}
assert paid_table.loc[2].to_dict() == {"Paid": 39, "Unpaid": 2}

display(work_table)
display((100 * work_table.div(work_table.sum(axis=1), axis=0)).round(1))
print({
    "eligible_n": len(f_eligible),
    "bias_corrected_cramers_v": work_effect,
    "raw_p": work_p,
    "targeted_five_bh": targeted_work_type_bh,
    "sensitivity_existing_f_family_bh": full_profile_sensitivity_bh,
    "sparse_expected_share": work_sparse_share,
})

## Plain-English dimension summary

In [7]:
print("\nMOST IMPORTANT POST-CLUSTER FINDINGS BY RETAINED DIMENSION")
print("=" * 72)
for line in plain_dimension_summaries:
    print("-", line)
print("\nThese are descriptive additional characteristics, not programme recommendations.")


MOST IMPORTANT POST-CLUSTER FINDINGS BY RETAINED DIMENSION
- D — Motivation and study experience: strongest additional differences concerned Useful employment/training contacts.
- D — Economic resources and household pressure: strongest additional differences concerned Ideal sector, Current or preferred field of study, Current institution type.
- D — Educational trajectory and cultural capital: strongest additional differences concerned Already searching for work/business, Primary personal income source, Main education funding source.
- E — Education usefulness and further-learning orientation: strongest additional differences concerned Job-acceptance conditions, Helpfulness of job-search contacts, Household size.
- F — Employment security and formality: strongest additional differences concerned Monthly work-income band, Primary personal income source, Hopes to change job.
- G — Household economic role and dependency: strongest additional differences concerned Useful employment/train